# Exploración Inicial — Señales EEG/EMG

Este notebook analiza el dataset `datos_eeg_emg.csv` generado desde `Data.txt`.

**Estructura del dataset:**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `Tiempo_s` | float | Vector de tiempo en segundos (0.001 s de paso → 1000 Hz) |
| `EEG_1` | float | Canal EEG principal |
| `EEG_2` | float | Canal EEG secundario |
| `EMG_1..4` | float | Canales EMG musculares (4 canales) |

**Canales clave según el pipeline:**
- Para detección de **marcadores de movimiento**: un canal **EMG** (burst detector)
- Para análisis de **BP y desincronización beta**: los canales **EEG**

> **Nota reunión 09-04:** el pipeline es → detectar bursts EMG → usar como marcadores → cortar EEG → buscar BP (onda lenta pre-movimiento) y caída en banda Beta (13–30 Hz)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.stats import kurtosis, skew
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 4)

DATA_PATH = '../data/datos_eeg_emg.csv'
SRATE = 1000  # Hz

## 1. Carga y descripción general

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Muestras  : {len(df):,}')
print(f'Columnas  : {list(df.columns)}')
print(f'Duración  : {df.Tiempo_s.iloc[-1]:.1f} s  ({df.Tiempo_s.iloc[-1]/60:.1f} min)')
print(f'Frec.     : {SRATE} Hz')
print()

df.describe().round(4)

## 2. Señal completa — todos los canales

Vista rápida para detectar artefactos, saturaciones o canales ruidosos.

In [ ]:
eeg_cols = ['EEG_1', 'EEG_2']
emg_cols = ['EMG_1', 'EMG_2', 'EMG_3', 'EMG_4']
t = df['Tiempo_s'].values

fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)

colors_eeg = ['#1f77b4', '#ff7f0e']
colors_emg = ['#2ca02c', '#d62728', '#9467bd', '#8c564b']

for i, col in enumerate(eeg_cols):
    axes[i].plot(t, df[col].values, color=colors_eeg[i], lw=0.4)
    axes[i].set_ylabel(col, fontsize=9)
    axes[i].set_title(f'{col}  —  EEG  |  rango: [{df[col].min():.3f}, {df[col].max():.3f}]', fontsize=8)

for j, col in enumerate(emg_cols):
    axes[j+2].plot(t, df[col].values, color=colors_emg[j], lw=0.4)
    axes[j+2].set_ylabel(col, fontsize=9)
    axes[j+2].set_title(f'{col}  —  EMG  |  rms: {np.sqrt(np.mean(df[col].values**2)):.5f}', fontsize=8)

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle('Vista completa — EEG (filas 1-2) y EMG (filas 3-6)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Estadísticas por canal

Comparar RMS, rango, kurtosis y skewness para identificar cuáles canales tienen mejor señal.

In [ ]:
signal_cols = eeg_cols + emg_cols
stats = []
for col in signal_cols:
    x = df[col].values
    stats.append({
        'Canal'    : col,
        'Tipo'     : 'EEG' if col.startswith('EEG') else 'EMG',
        'Media'    : np.mean(x),
        'Std'      : np.std(x),
        'RMS'      : np.sqrt(np.mean(x**2)),
        'Min'      : np.min(x),
        'Max'      : np.max(x),
        'Rango'    : np.max(x) - np.min(x),
        'Kurtosis' : kurtosis(x),
        'Skewness' : skew(x),
    })

stats_df = pd.DataFrame(stats).set_index('Canal')
stats_df.round(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = colors_eeg + colors_emg

stats_df['RMS'].plot(kind='bar', ax=axes[0], color=colors, title='RMS por canal')
axes[0].set_ylabel('RMS (µV)')
axes[0].tick_params(axis='x', rotation=45)

stats_df['Rango'].plot(kind='bar', ax=axes[1], color=colors, title='Rango (max−min) por canal')
axes[1].set_ylabel('Amplitud (µV)')
axes[1].tick_params(axis='x', rotation=45)

stats_df['Kurtosis'].plot(kind='bar', ax=axes[2], color=colors, title='Kurtosis por canal')
axes[2].axhline(3, color='red', lw=1, ls='--', label='Gaussiana (K=3)')
axes[2].legend(fontsize=8)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Comparación estadística de canales', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Densidad espectral de potencia (PSD)

Ver en qué bandas concentra energía cada canal. Para EEG nos interesan las **bandas alfa (8–13 Hz) y beta (13–30 Hz)**. Alta kurtosis en EMG indica bursts musculares.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# EEG PSD
for i, col in enumerate(eeg_cols):
    x = df[col].values
    freqs, psd = signal.welch(x, SRATE, nperseg=2048)
    axes[0].semilogy(freqs, psd, label=col, lw=1.2, color=colors_eeg[i])

bandas = [('Delta', 0.5, 4, '#cce5ff'), ('Theta', 4, 8, '#d4edda'),
          ('Alpha', 8, 13, '#fff3cd'), ('Beta', 13, 30, '#f8d7da'), ('Gamma', 30, 100, '#e2d9f3')]
for nombre, lo, hi, color in bandas:
    axes[0].axvspan(lo, hi, alpha=0.15, color=color, label=nombre)

axes[0].set_xlim(0, 120)
axes[0].set_title('PSD — Canales EEG')
axes[0].set_ylabel('PSD (µV²/Hz)')
axes[0].legend(fontsize=8, ncol=4)
axes[0].grid(True, alpha=0.3)

# EMG PSD
for j, col in enumerate(emg_cols):
    x = df[col].values
    freqs, psd = signal.welch(x, SRATE, nperseg=2048)
    axes[1].semilogy(freqs, psd, label=col, lw=1, color=colors_emg[j])

axes[1].set_xlim(0, 300)
axes[1].set_title('PSD — Canales EMG')
axes[1].set_xlabel('Frecuencia (Hz)')
axes[1].set_ylabel('PSD (µV²/Hz)')
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Zoom: primeros 10 segundos

Ver la morfología de la señal a nivel de muestra para detectar ruido de línea (50 Hz), artefactos de movimiento o saturaciones.

In [ ]:
ZOOM_S = 10  # segundos a visualizar
n_zoom = int(ZOOM_S * SRATE)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for i, col in enumerate(eeg_cols):
    axes[0].plot(t[:n_zoom], df[col].values[:n_zoom], label=col, lw=0.8, color=colors_eeg[i])
axes[0].set_title(f'EEG — primeros {ZOOM_S} s')
axes[0].set_ylabel('Amplitud (µV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j, col in enumerate(emg_cols):
    axes[1].plot(t[:n_zoom], df[col].values[:n_zoom], label=col, lw=0.6, color=colors_emg[j], alpha=0.8)
axes[1].set_title(f'EMG — primeros {ZOOM_S} s')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud (µV)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. ¿Cuál canal EMG usar como detector de bursts?

Evaluamos cuál de los 4 canales EMG tiene mayor varianza y kurtosis.

Un buen canal para burst detection tiene:
- **Alta kurtosis** → distribución con colas pesadas (eventos de alta amplitud bien definidos)
- **RMS alto relativo a la línea base** → buena relación señal/ruido
- **PSD con energía en 20–150 Hz** → banda típica del EMG muscular

In [ ]:
emg_ranking = []
for col in emg_cols:
    x = df[col].values
    x_centered = x - np.mean(x)
    x_rect = np.abs(x_centered)
    
    # Energía en banda EMG (20–150 Hz)
    freqs, psd = signal.welch(x_centered, SRATE, nperseg=1024)
    mask_emg = (freqs >= 20) & (freqs <= 150)
    emg_band_power = np.trapz(psd[mask_emg], freqs[mask_emg])
    total_power = np.trapz(psd, freqs)
    emg_ratio = emg_band_power / total_power
    
    emg_ranking.append({
        'Canal'           : col,
        'RMS'             : np.sqrt(np.mean(x_centered**2)),
        'Kurtosis'        : kurtosis(x_rect),
        'Potencia_EMG_band': emg_band_power,
        'Ratio_EMG_band'  : emg_ratio,
        'Score'           : kurtosis(x_rect) * emg_ratio * 100,  # score compuesto
    })

ranking_df = pd.DataFrame(emg_ranking).set_index('Canal').sort_values('Score', ascending=False)
print('Ranking de canales EMG para detección de bursts:')
print('(mayor Score = mejor candidato)\n')
ranking_df.round(4)

In [ ]:
best_emg = ranking_df.index[0]
print(f'Canal EMG recomendado para burst detection: {best_emg}')

# Visualizar el mejor canal rectificado
x_best = df[best_emg].values
x_best = x_best - np.mean(x_best)
x_rect = np.abs(x_best)
# Normalizar entre 0 y 1 (como hace app_edu.py)
x_norm = (x_rect - x_rect.min()) / (x_rect.max() - x_rect.min())

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(t, x_best, lw=0.4, color='steelblue', label=f'{best_emg} (centrado)')
axes[0].set_title(f'Canal EMG más activo: {best_emg}')
axes[0].set_ylabel('Amplitud (µV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, x_norm, lw=0.4, color='darkorange', label='Rectificado y normalizado [0,1]')
axes[1].axhline(0.2, color='red', lw=1.2, ls='--', label='Umbral ejemplo (0.2)')
axes[1].set_title('Señal lista para umbralización (burst detection)')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud norm.')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. EEG: verificar bandas de interés

Filtrar y visualizar la **banda Beta (13–30 Hz)** en EEG_1 y EEG_2. Esta es la banda clave para detectar **desincronización pre-movimiento** (ERDS).

In [ ]:
from scipy.signal import butter, sosfilt, sosfiltfilt

def bandpass(x, lo, hi, srate, order=4):
    nyq = srate / 2
    sos = butter(order, [lo/nyq, hi/nyq], btype='band', output='sos')
    return sosfiltfilt(sos, x)

bands = [
    ('Alpha (8–13 Hz)', 8, 13),
    ('Beta (13–30 Hz)', 13, 30),
]

# 2 EEG × 2 bandas
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)

for row, (band_name, lo, hi) in enumerate(bands):
    for col_i, eeg_col in enumerate(eeg_cols):
        x = df[eeg_col].values
        x_filt = bandpass(x, lo, hi, SRATE)
        axes[row][col_i].plot(t, x_filt, lw=0.5, color=colors_eeg[col_i], alpha=0.85)
        axes[row][col_i].set_title(f'{eeg_col} — {band_name}', fontsize=9)
        axes[row][col_i].set_ylabel('Amplitud (µV)', fontsize=8)
        axes[row][col_i].grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Tiempo (s)')

plt.suptitle('EEG filtrado por bandas de interés clínico', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Espectrograma EEG — vista tiempo-frecuencia

Ver si hay modulación espectral a lo largo del tiempo (indicativo de actividad cognitiva/motora).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(eeg_cols):
    x = df[col].values
    f, t_spec, Sxx = signal.spectrogram(x, SRATE, nperseg=512, noverlap=256)
    freq_mask = f <= 80
    im = axes[i].pcolormesh(t_spec, f[freq_mask], 10*np.log10(Sxx[freq_mask] + 1e-12),
                             shading='gouraud', cmap='viridis')
    axes[i].set_title(f'Espectrograma {col}  —  EEG')
    axes[i].set_xlabel('Tiempo (s)')
    axes[i].set_ylabel('Frecuencia (Hz)')
    for freq_line, label in [(8,'\u03b1'), (13,'\u03b2'), (30,'\u03b3')]:
        axes[i].axhline(freq_line, color='white', lw=0.8, ls='--', alpha=0.6)
        axes[i].text(t_spec[-1]*1.01, freq_line, label, color='white', fontsize=8, va='center')
    plt.colorbar(im, ax=axes[i], label='dB')

plt.suptitle('Espectrograma EEG (0–80 Hz)', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Resumen y recomendaciones

Basado en el análisis anterior:

In [ ]:
print('=' * 55)
print('RESUMEN DE EXPLORACIÓN')
print('=' * 55)
print(f'Duración total   : {df.Tiempo_s.iloc[-1]:.0f} s  ({df.Tiempo_s.iloc[-1]/60:.1f} min)')
print(f'Tasa de muestreo : {SRATE} Hz')
print(f'Muestras         : {len(df):,}')
print()
print('Canales EEG:')
for col in eeg_cols:
    print(f'  {col}: rango [{df[col].min():.3f}, {df[col].max():.3f}]')
print()
print('Canales EMG:')
for col in emg_cols:
    print(f'  {col}: rms={np.sqrt(np.mean(df[col].values**2)):.5f}  '
          f'rango [{df[col].min():.4f}, {df[col].max():.4f}]')
print()
print('Canal EMG recomendado para burst detection:')
print(f'  → {best_emg}  (mayor kurtosis y energía en banda 20–150 Hz)')
print()
print('Próximos pasos:')
print('  1. Aplicar filtros (highpass 1 Hz, lowpass 100 Hz, notch 50 Hz)')
print('  2. Comparar BacAV vs clustering espectral (Sección 10)')
print('  3. Usar bursts como marcadores → segmentar EEG')
print('  4. Calcular ERP y ERDS (desincronización beta pre-movimiento)')
print('=' * 55)

## 10. Comparación: detección de bursts EMG — BacAV vs Clustering Espectral

Comparamos dos enfoques para detectar el **onset de bursts musculares** en el canal EMG óptimo:

| Método | Principio |
|--------|-----------|
| **BacAV** (Vial et al., 2020) | Umbralización sobre señal rectificada y normalizada [0,1]; verifica amplitud media en ventana pre/post candidato |
| **K-Means espectral** | Extrae features por ventana corta (RMS + potencia 20–150 Hz) y agrupa en `burst` vs `reposo` |

La congruencia entre ambos valida que los eventos detectados corresponden a activaciones musculares reales.

In [ ]:
# ── 1. Canal EMG seleccionado ─────────────────────────────────────────────────
EMG_CH  = best_emg
emg_raw = df[EMG_CH].values
t       = df['Tiempo_s'].values
print(f'Canal EMG seleccionado: {EMG_CH}')

# ── 2. Método BacAV ───────────────────────────────────────────────────────────
# Reproducción del algoritmo descrito en Vial et al. (2020):
# rectificar → normalizar [0,1] → umbralizar → verificar ventana pre/post

def detect_bursts_bacav(
    x, srate,
    threshold=0.20,
    time_before=0.10,
    time_after=0.10,
    amp_before=0.05,
    amp_after=0.15,
    burst_duration=0.20
):
    x_c    = x - np.mean(x)
    x_rect = np.abs(x_c)
    x_norm = (x_rect - x_rect.min()) / (x_rect.max() - x_rect.min())
    n_before = int(time_before * srate)
    n_after  = int(time_after  * srate)
    n_gap    = int(burst_duration * srate)
    bursts, last = [], -n_gap
    for idx in np.where(x_norm > threshold)[0]:
        if idx - last < n_gap:
            continue
        if idx < n_before or idx + n_after >= len(x_norm):
            continue
        if (np.mean(x_norm[idx - n_before:idx]) < amp_before and
                np.mean(x_norm[idx:idx + n_after]) > amp_after):
            bursts.append(idx)
            last = idx
    return np.array(bursts)

bursts_bacav = detect_bursts_bacav(emg_raw, SRATE)
print(f'BacAV   → {len(bursts_bacav):4d} bursts')

In [ ]:
from sklearn.cluster      import KMeans
from sklearn.preprocessing import StandardScaler

# ── 3. Método K-Means espectral ───────────────────────────────────────────────
WIN_S = 0.050
HOP_S = 0.010
n_win = int(WIN_S * SRATE)
n_hop = int(HOP_S * SRATE)

x_c    = emg_raw - np.mean(emg_raw)
x_rect = np.abs(x_c)

feat_list, time_list = [], []
for start in range(0, len(x_c) - n_win, n_hop):
    seg      = x_c[start:start + n_win]
    rms      = float(np.sqrt(np.mean(np.abs(seg)**2)))
    fw, pw   = signal.welch(seg, SRATE, nperseg=n_win)
    m        = (fw >= 20) & (fw <= 150)
    band_pow = float(np.trapz(pw[m], fw[m])) if m.sum() > 1 else 0.0
    feat_list.append([rms, band_pow])
    time_list.append((start + n_win // 2) / SRATE)

feat_arr = np.array(feat_list)
time_arr = np.array(time_list)

feat_sc       = StandardScaler().fit_transform(feat_arr)
labels_km     = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(feat_sc)
burst_cluster = int(np.argmax([feat_arr[labels_km == i, 0].mean() for i in range(2)]))
burst_mask_km = labels_km == burst_cluster

transitions   = np.where(np.diff(burst_mask_km.astype(int)) == 1)[0]
bursts_kmeans = (time_arr[transitions] * SRATE).astype(int)
print(f'K-Means → {len(bursts_kmeans):4d} bursts')
print(f'Diferencia: {abs(len(bursts_bacav) - len(bursts_kmeans))} bursts')

In [ ]:
# ── 4. Visualización comparativa (primeros 30 s) ──────────────────────────────
SHOW_S   = 30
n_show   = int(SHOW_S * SRATE)

x_c_full    = emg_raw - np.mean(emg_raw)
x_rect_full = np.abs(x_c_full)
x_norm_full = (x_rect_full - x_rect_full.min()) / (x_rect_full.max() - x_rect_full.min())

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

axes[0].plot(t[:n_show], x_c_full[:n_show], lw=0.4, color='steelblue')
axes[0].set_title(f'{EMG_CH}  —  señal EMG centrada', fontsize=10)
axes[0].set_ylabel('Amplitud', fontsize=9)
axes[0].grid(True, alpha=0.25)

bv_show = bursts_bacav[bursts_bacav < n_show]
axes[1].plot(t[:n_show], x_norm_full[:n_show], lw=0.4, color='steelblue', alpha=0.6, label='norm.')
axes[1].axhline(0.20, color='gray', lw=0.8, ls='--', label='threshold')
axes[1].vlines(t[bv_show], 0, 1, color='crimson', lw=0.8, alpha=0.7, label=f'BacAV ({len(bv_show)})')
axes[1].set_title('Método BacAV  —  umbralización sobre señal rectificada', fontsize=10)
axes[1].set_ylabel('Norm. [0,1]', fontsize=9)
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.25)
axes[1].set_ylim(-0.05, 1.1)

burst_signal = np.zeros(n_show)
for i, ts in enumerate(time_arr):
    idx = int(ts * SRATE)
    if idx >= n_show:
        break
    s = max(0, idx - n_win // 2)
    e = min(n_show, idx + n_win // 2)
    if burst_mask_km[i]:
        burst_signal[s:e] = 1

km_show = bursts_kmeans[bursts_kmeans < n_show]
axes[2].fill_between(t[:n_show], 0, burst_signal, alpha=0.25, color='green', label='zona burst')
axes[2].plot(t[:n_show], x_norm_full[:n_show], lw=0.4, color='steelblue', alpha=0.6)
axes[2].vlines(t[km_show], 0, 1, color='darkgreen', lw=0.8, alpha=0.8, label=f'K-Means ({len(km_show)})')
axes[2].set_title('Método K-Means espectral  —  RMS + potencia 20–150 Hz por ventana', fontsize=10)
axes[2].set_xlabel('Tiempo (s)')
axes[2].set_ylabel('Norm. [0,1]', fontsize=9)
axes[2].legend(fontsize=8, loc='upper right')
axes[2].grid(True, alpha=0.25)
axes[2].set_ylim(-0.05, 1.1)

plt.suptitle(f'Comparación BacAV vs K-Means  —  {EMG_CH}  (primeros {SHOW_S} s)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Concordancia cuantitativa (±200 ms) ────────────────────────────────────
TOL_S = 0.200

matched_bv = sum(
    1 for b in bursts_bacav if np.any(np.abs(bursts_kmeans - b) / SRATE <= TOL_S)
)
matched_km = sum(
    1 for b in bursts_kmeans if np.any(np.abs(bursts_bacav - b) / SRATE <= TOL_S)
)

prec_bv = matched_bv / len(bursts_bacav)  * 100 if len(bursts_bacav)  else 0
recall  = matched_km / len(bursts_kmeans) * 100 if len(bursts_kmeans) else 0

print(f'Tolerancia: ±{int(TOL_S*1000)} ms')
print(f'Bursts BacAV  con match K-Means : {matched_bv:4d}/{len(bursts_bacav)}  ({prec_bv:.1f}%)')
print(f'Bursts K-Means con match BacAV  : {matched_km:4d}/{len(bursts_kmeans)}  ({recall:.1f}%)')

fig, ax = plt.subplots(figsize=(16, 3))
ax.scatter(t[bursts_bacav],  np.ones(len(bursts_bacav)),   marker='|', s=60,
           color='crimson',   label=f'BacAV ({len(bursts_bacav)})',   lw=1.2)
ax.scatter(t[bursts_kmeans], np.ones(len(bursts_kmeans))*1.5, marker='|', s=60,
           color='darkgreen', label=f'K-Means ({len(bursts_kmeans)})', lw=1.2)
ax.set_yticks([1, 1.5])
ax.set_yticklabels(['BacAV', 'K-Means'], fontsize=9)
ax.set_xlabel('Tiempo (s)')
ax.set_xlim(0, t[-1])
ax.set_title('Distribución temporal de bursts detectados — EMG', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()